# Incremental Candidate Detection for Rainfall-Triggered Landslides

**Purpose:** Reads the GeoTIFFs produced by `extracting_data.ipynb` (single-date
same-day S2 mosaic + DEM + optional S1 SAR) from the HuggingFace dataset and
generates candidate landslide ROI masks via unsupervised change detection.
See `landslide_workflow.md` Stage 1.1 for the methodology.

**Incremental improvement:** This notebook first queries HuggingFace for existing
candidate results (`candidates/incident_<ID>_candidates.json`), then only processes
incidents that have imagery on the hub but no candidates yet. This makes it safe
to re-run as new imagery is added.

**Per-incident pipeline:**
1. Pull `incident_{id}_before.tif`, `_after.tif`, `_slope.tif`, `_aspect.tif`,
   and optional `_sar_pre.tif` / `_sar_post.tif` from HuggingFace
2. Compute change indices: dNDVI, dNDWI, dBSI, dNBR (post − pre)
3. SAR amplitude change (VV) from paired pre/post S1
4. Fuse changes into a binary mask, gate by slope (DEM, workflow Stage 1.1)
5. Morphological cleanup, connected-component blob extraction + area/elongation filter
6. Clip candidate image chips and upload everything to HuggingFace

## Configuration

All paths, thresholds, and HuggingFace credentials are set here.

In [1]:
# %% [code]
# --------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------
import os, json, re, shutil, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.mask import mask as rio_mask
from shapely.geometry import box
from skimage import measure, morphology
from huggingface_hub import hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient

# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
IMAGERY_REPO = "sasudo2/landslides"
CANDIDATE_REPO = "sasudo2/landslide_data"
REPO_TYPE = "dataset"
DATASET_REVISION = "main"

CANDIDATE_DIR = "/kaggle/working/candidates"
CLIPPED_DIR = "/kaggle/working/clipped_images"
JSONL_PATH = f"{CANDIDATE_DIR}/candidates.jsonl"
JSONL_LOCK = threading.Lock()
PROCESS_ALLOWED = threading.Event()
PROCESS_ALLOWED.set()
MAX_WORKERS = 4
os.makedirs(CANDIDATE_DIR, exist_ok=True)
os.makedirs(CLIPPED_DIR, exist_ok=True)

# Band indices (0-based) into the 13-band S2 array [B1..B12, SCL]
B = {
    'B2': 1, 'B3': 2, 'B4': 3, 'B8': 7, 'B11': 10, 'B12': 11,
}

# Change-detection thresholds (tuned for rainfall-triggered slides)
DNDVI_THRESH   = -0.15   # vegetation loss (tighter)
DNDWI_THRESH   = -0.08   # moisture / bare-soil exposure (tighter)
DBSI_THRESH    =  0.12   # bare-soil increase (tighter)
DNBR_THRESH    = -0.15   # vegetation stripping (tighter)
VOTE_THRESHOLD = 2       # at least 2 indices must agree
MIN_SLOPE_DEG  = 20       # skip change on flat terrain
MIN_BLOB_AREA_M2 = 2_000
MAX_BLOB_AREA_M2 = 2_000_000
MAX_ELONGATION = 6.0
MAX_CANDIDATES_PER_INCIDENT = 20
CHIP_SIZE_M = 1280      # fixed square chip size for ML input (1280m x 1280m = 128x128 px @ 10m)

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
api = HfApi(token=hf_token)
print("Configuration loaded.")

Configuration loaded.


## Step 1: Determine which incidents already have candidates

We query HuggingFace for two things:
1. **Incidents with imagery:** folders named `incident_<ID>/` in the repo root
2. **Incidents with candidates:** files named `candidates/incident_<ID>_candidates.json`

Only incidents that have imagery but **no** candidates yet will be processed.

In [2]:
# %% [code]
print("\n=== Querying HuggingFace for existing imagery and candidates ===")

incidents_with_imagery = set()
incidents_with_candidates = set()

# Query imagery repo for incident folders
try:
    repo_files = api.list_repo_files(
        repo_id=IMAGERY_REPO, repo_type=REPO_TYPE, revision=DATASET_REVISION)
    for fpath in repo_files:
        m = re.match(r'incident_(\d+)/', fpath)
        if m:
            incidents_with_imagery.add(int(m.group(1)))
except Exception as e:
    print(f"Could not list imagery repo: {e}")

# Download existing JSONL from candidate repo to identify uploaded incidents
try:
    hf_hub_download(
        repo_id=CANDIDATE_REPO, repo_type=REPO_TYPE, revision=DATASET_REVISION,
        filename="candidates.jsonl", local_dir=CANDIDATE_DIR, token=hf_token)
    with open(JSONL_PATH) as f:
        for line in f:
            rec = json.loads(line)
            incidents_with_candidates.add(int(rec["incident_id"]))
    print(f"Loaded existing candidates.jsonl: {len(incidents_with_candidates)} incidents")
except Exception:
    print("No existing candidates.jsonl, starting fresh.")

to_process = sorted(incidents_with_imagery - incidents_with_candidates)

print(f"Incidents with imagery on HF: {len(incidents_with_imagery)}")
print(f"Incidents with candidates on HF: {len(incidents_with_candidates)}")
print(f"To process (imagery but no candidates): {len(to_process)}")
if to_process:
    print(f"First 5: {to_process[:5]}")


=== Querying HuggingFace for existing imagery and candidates ===
No existing candidates.jsonl, starting fresh.
Incidents with imagery on HF: 3032
Incidents with candidates on HF: 0
To process (imagery but no candidates): 3032
First 5: [36869, 37032, 37136, 37717, 37879]


## Step 2: HuggingFace raster fetching helpers

Download the pre-computed GeoTIFFs for a given incident from the dataset repo.

In [3]:
# %% [code]
def _fetch(incident_id, label):
    """Download a single GeoTIFF from HuggingFace. Returns local path or None."""
    incident_id = int(incident_id)
    remote_path = f"incident_{incident_id}/incident_{incident_id}_{label}.tif"
    try:
        return hf_hub_download(repo_id=IMAGERY_REPO, repo_type=REPO_TYPE,
                               revision=DATASET_REVISION,
                               filename=remote_path, token=hf_token)
    except Exception as e:
        print(f"   _fetch failed for {remote_path}: {type(e).__name__}: {e}")
        return None

def fetch_incident_rasters(incident_id):
    """Download all available rasters for an incident.
    Returns dict of {label: local_path} or None if mandatory files are missing.
    Mandatory: before, after, slope. Optional: aspect, sar_pre, sar_post.
    """
    paths = {}
    for label in ("before", "after", "slope"):
        p = _fetch(incident_id, label)
        if p is None:
            print(f"   Missing mandatory {label} for incident {incident_id}.")
            return None
        paths[label] = p
    paths["aspect"] = _fetch(incident_id, "aspect")
    paths["sar_pre"] = _fetch(incident_id, "sar_pre")
    paths["sar_post"] = _fetch(incident_id, "sar_post")
    return paths

## Step 3: Change-index computation and candidate mask generation

Compute NDVI, NDWI, BSI, NBR for pre and post, derive change layers,
fuse with SAR amplitude change (VV), gate by slope, and build a binary
candidate mask. See `landslide_workflow.md` Stage 1.1 for the methodology.

In [4]:
# %% [code]
def _safe_div(num, den):
    num = num.astype(np.float32)
    den = den.astype(np.float32)
    out = np.full_like(num, np.nan)
    safe = np.abs(den) >= 1e-6
    out[safe] = num[safe] / den[safe]
    return out

def compute_indices(arr):
    """Compute NDVI, NDWI, BSI, NBR from a 13-band S2 array (B1..B12, SCL)."""
    b2, b3, b4, b8, b11, b12 = (arr[B["B2"]], arr[B["B3"]], arr[B["B4"]],
                                arr[B["B8"]], arr[B["B11"]], arr[B["B12"]])
    b2 = b2.astype(np.float32); b3 = b3.astype(np.float32)
    b4 = b4.astype(np.float32); b8 = b8.astype(np.float32)
    b11 = b11.astype(np.float32); b12 = b12.astype(np.float32)
    return {
        'NDVI': _safe_div(b8 - b4, b8 + b4),
        'NDWI': _safe_div(b3 - b8, b3 + b8),
        'BSI':  _safe_div((b11 + b4) - (b8 + b2), (b11 + b4) + (b8 + b2)),
        'NBR':  _safe_div(b8 - b12, b8 + b12),
    }

def build_change_mask(before_path, after_path, slope_path,
                      sar_pre_path=None, sar_post_path=None):
    """Build a binary candidate mask from pre/post S2, DEM slope, and optional S1 SAR.

    Returns:
        mask_bool (np.ndarray): True where change is detected
        vote_count (np.ndarray): per-pixel vote count (how many indices triggered)
        transform: affine transform of the mask
        crs: coordinate reference system
    """
    with rasterio.open(before_path) as src:
        before_arr = src.read()
        transform = src.transform
        crs = src.crs
        shape = (src.height, src.width)

    with rasterio.open(after_path) as src:
        after_arr = src.read(out_shape=(src.count, *shape),
                             resampling=Resampling.bilinear)

    idx_b = compute_indices(before_arr)
    idx_a = compute_indices(after_arr)

    dNDVI = idx_a["NDVI"] - idx_b["NDVI"]
    dNDWI = idx_a["NDWI"] - idx_b["NDWI"]
    dBSI  = idx_a["BSI"]  - idx_b["BSI"]
    dNBR  = idx_a["NBR"]  - idx_b["NBR"]

    # Vote-count fusion: at least VOTE_THRESHOLD indices must trigger
    votes = (
        (np.nan_to_num(dNDVI) <= DNDVI_THRESH).astype(np.int8) +
        (np.nan_to_num(dNBR)  <= DNBR_THRESH).astype(np.int8) +
        (np.nan_to_num(dBSI)  >= DBSI_THRESH).astype(np.int8) +
        (np.nan_to_num(dNDWI) <= DNDWI_THRESH).astype(np.int8)
    )

    # SAR backscatter ratio (VV channel) — counts as a 5th vote
    sar_change = None
    if sar_pre_path is not None and sar_post_path is not None:
        with rasterio.open(sar_pre_path) as src:
            pre_sar = src.read(indexes=[1], out_shape=(1, *shape),
                               resampling=Resampling.bilinear)[0].astype(np.float32)
        with rasterio.open(sar_post_path) as src:
            post_sar = src.read(indexes=[1], out_shape=(1, *shape),
                               resampling=Resampling.bilinear)[0].astype(np.float32)
        vv_diff = post_sar - pre_sar
        sar_change = vv_diff > 1.0
        votes = votes + sar_change.astype(np.int8)

    # Resample slope (30m) to 10m S2 grid
    with rasterio.open(slope_path) as src_s:
        slope_rs = np.empty(shape, dtype=np.float32)
        reproject(source=rasterio.band(src_s, 1), destination=slope_rs,
                  src_transform=src_s.transform, src_crs=src_s.crs,
                  dst_transform=transform, dst_crs=crs, resampling=Resampling.bilinear)

    slope_mask = slope_rs >= MIN_SLOPE_DEG
    combined = (votes >= VOTE_THRESHOLD) & slope_mask

    return combined, votes, transform, crs

## Step 4: Blob extraction and candidate clipping

Clean the binary mask, extract connected components, filter by area and
elongation, then clip fixed-size 1280m x 1280m chips (128x128 px @ 10m)
centered on each candidate, clamped to stay within incident image bounds
so they contain only real data.


In [5]:
# %% [code]
def extract_candidate_blobs(mask_bool, vote_count, transform, pixel_size_m=10):
    """Extract connected-component blobs from the change mask.
    Filters by area and elongation. Returns list sorted by mean vote strength (strongest first).
    """
    mask_bool = morphology.binary_dilation(mask_bool, morphology.disk(3))
    mask_bool = morphology.remove_small_objects(mask_bool, min_size=3)
    mask_bool = morphology.binary_closing(mask_bool, morphology.disk(1))

    labeled = measure.label(mask_bool, connectivity=2)
    candidates = []
    for region in measure.regionprops(labeled):
        area_m2 = region.area * (pixel_size_m ** 2)
        if area_m2 < MIN_BLOB_AREA_M2 or area_m2 > MAX_BLOB_AREA_M2:
            continue
        major = region.major_axis_length or 1
        minor = region.minor_axis_length or 1
        elongation = major / max(minor, 1)
        if elongation > MAX_ELONGATION:
            continue
        min_row, min_col, max_row, max_col = region.bbox
        lon_min, lat_max = transform * (min_col, min_row)
        lon_max, lat_min = transform * (max_col, max_row)
        # Mean vote strength over the blob pixels
        blob_mask = labeled[region.slice] == region.label
        blob_votes = vote_count[region.slice][blob_mask]
        strength = float(np.mean(blob_votes)) if blob_votes.size > 0 else 0.0
        candidates.append({
            "area_m2": area_m2,
            "elongation": elongation,
            "strength": strength,
            "bbox_lonlat": [lon_min, lat_min, lon_max, lat_max],
        })
    candidates.sort(key=lambda c: c["strength"], reverse=True)
    return candidates

def fixed_size_chip_bbox(center_lon, center_lat, chip_size_m):
    """Create a fixed-size square chip bbox centered on (center_lon, center_lat).
    chip_size_m: side length in meters (chip will be chip_size_m x chip_size_m).
    Returns [lon_min, lat_min, lon_max, lat_max].
    """
    half = chip_size_m / 2.0
    m_per_deg_lat = 111_320
    m_per_deg_lon = 111_320 * np.cos(np.radians(center_lat))
    dlat = half / m_per_deg_lat
    dlon = half / m_per_deg_lon
    return [center_lon - dlon, center_lat - dlat,
            center_lon + dlon, center_lat + dlat]


def clip_raster_to_bbox(src_path, bbox_lonlat, dst_path, indexes=None, tags=None):
    """Clip a raster to a bounding box and write selected bands to dst_path.
    indexes: list of 1-based band indices to include (e.g. [2,3,4] for B2/B3/B4).
             If None, reads all bands.
    tags: optional dict of metadata tags to embed in the GeoTIFF
          (e.g. {"INCIDENT_ID": "12345", "CANDIDATE_INDEX": "1"}).
    """
    lon_min, lat_min, lon_max, lat_max = bbox_lonlat
    geom = box(lon_min, lat_min, lon_max, lat_max)
    with rasterio.open(src_path) as src:
        if indexes is None:
            out_image, out_transform = rio_mask(src, [geom], crop=True, all_touched=True)
        else:
            out_image, out_transform = rio_mask(src, [geom], crop=True, all_touched=True,
                                               indexes=indexes)
    if out_image.shape[-1] == 0 or out_image.shape[-2] == 0:
        print(f"  clip skipped (no overlap): {dst_path}")
        return None

    with rasterio.open(src_path) as src2:
        out_meta = src2.meta.copy()
    if indexes is None:
        band_count = src2.count
    elif isinstance(indexes, int):
        band_count = 1
    else:
        band_count = len(indexes)
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[-2],
        "width": out_image.shape[-1],
        "transform": out_transform,
        "count": band_count,
    })
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    if out_image.ndim == 2:
        out_image = np.expand_dims(out_image, axis=0)
    with rasterio.open(dst_path, "w", **out_meta) as dst:
        dst.write(out_image)
        if tags:
            dst.update_tags(**tags)
    return dst_path
def clamp_bbox_to_bounds(bbox_lonlat, bounds):
    """Shift a fixed-size bbox so it stays entirely within raster bounds.
    bbox_lonlat: [lon_min, lat_min, lon_max, lat_max]
    bounds: rasterio bounds object (left, bottom, right, top)
    Returns clamped [lon_min, lat_min, lon_max, lat_max].
    """
    lon_min, lat_min, lon_max, lat_max = bbox_lonlat
    half_lon = (lon_max - lon_min) / 2
    half_lat = (lat_max - lat_min) / 2
    cx = (lon_min + lon_max) / 2
    cy = (lat_min + lat_max) / 2
    # Shift horizontally if outside bounds
    if lon_min < bounds.left:
        cx += bounds.left - lon_min
    if lon_max > bounds.right:
        cx -= lon_max - bounds.right
    # Shift vertically if outside bounds
    if lat_min < bounds.bottom:
        cy += bounds.bottom - lat_min
    if lat_max > bounds.top:
        cy -= lat_max - bounds.top
    return [cx - half_lon, cy - half_lat, cx + half_lon, cy + half_lat]


## Step 5: Process unchecked incidents (parallel) and upload (batch)

Incidents are processed in parallel using `ThreadPoolExecutor`. After all are done,
clipped chips and `candidates.jsonl` are uploaded in two batch calls to HuggingFace.

In [6]:
def _flush_pending(pending):
    """Upload a batch of incidents with a MINIMAL number of API commits.

    Atomic move (same filesystem) pending folders to a staging directory,
    upload staging in ONE commit, plus ONE commit for candidates.jsonl,
    then delete staging.  Background threads writing to CLIPPED_DIR during
    the upload cannot affect the staging directory, so partial or duplicate
    uploads are avoided.
    """
    if not pending:
        return
    total = sum(n for _, n in pending)
    staging = f"{CANDIDATE_DIR}/_staging"
    PROCESS_ALLOWED.clear()
    try:
        # Atomically move pending folders out of CLIPPED_DIR
        if os.path.exists(staging):
            shutil.rmtree(staging, ignore_errors=True)
        os.makedirs(staging, exist_ok=True)
        for inc_id, _ in pending:
            src = f"{CLIPPED_DIR}/incident_{inc_id}"
            if os.path.exists(src):
                shutil.move(src, os.path.join(staging, f"incident_{inc_id}"))

        # Zip the entire batch into a single archive
        inc_ids = [str(inc_id) for inc_id, _ in pending]
        zip_name = f"incidents_{inc_ids[0]}_{inc_ids[-1]}.zip"
        zip_path = os.path.join(CANDIDATE_DIR, zip_name)
        shutil.make_archive(zip_path.replace('.zip', ''), 'zip', staging)

        # ONE commit for the zip archive
        api.upload_file(
            path_or_fileobj=zip_path,
            path_in_repo=zip_name,
            repo_id=CANDIDATE_REPO, repo_type=REPO_TYPE, revision=DATASET_REVISION)

        # ONE commit for the JSONL
        api.upload_file(
            path_or_fileobj=JSONL_PATH,
            path_in_repo="candidates.jsonl",
            repo_id=CANDIDATE_REPO, repo_type=REPO_TYPE, revision=DATASET_REVISION)

        shutil.rmtree(staging, ignore_errors=True)
        os.remove(zip_path)
        print(f"  Flushed batch: {len(pending)} incidents, {total} candidates ({zip_name})")
    except Exception as e:
        print(f"  Batch upload failed: {e}")
        # Restore moved folders back to CLIPPED_DIR on failure
        if os.path.exists(staging):
            for item in os.listdir(staging):
                shutil.move(os.path.join(staging, item),
                            os.path.join(CLIPPED_DIR, item))
            shutil.rmtree(staging, ignore_errors=True)
    finally:
        PROCESS_ALLOWED.set()


def process_one(inc_id):
    """Process a single incident. Returns (incident_id, n_candidates)."""
    PROCESS_ALLOWED.wait()
    try:
        result = find_candidates_for_incident(inc_id)
    except Exception as e:
        print(f"Incident {inc_id} failed: {e}")
        return (inc_id, 0)
    if result is None:
        return (inc_id, 0)
    return (inc_id, len(result))


if not to_process:
    print("\n=== No unchecked incidents. All imagery has candidates. ===")
else:
    print(f"\n=== Processing {len(to_process)} incidents without candidates ===\n")
    
    pending = []
    done = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_one, inc_id): inc_id for inc_id in to_process}
        for future in as_completed(futures):
            inc_id = futures[future]
            _, n = future.result()
            done += 1
            if n > 0:
                pending.append((inc_id, n))
                # Flush every BATCH_SIZE incidents
                if len(pending) >= BATCH_SIZE:
                    print(f"  [{done}/{len(to_process)} done] Reached {len(pending)} incidents \u2014 flushing batch")
                    _flush_pending(pending)
                    pending = []
            if done % 50 == 0 or done == len(to_process):
                print(f"  Progress: {done}/{len(to_process)} incidents processed")
    
    # Flush any remaining
    if pending:
        print(f"  Flushing final batch ({len(pending)} incidents)")
        _flush_pending(pending)
    
    print("\n=== Done ===")


=== Processing 3032 incidents without candidates ===

Incident 36869 failed: name 'find_candidates_for_incident' is not defined
Incident 37032 failed: name 'find_candidates_for_incident' is not defined
Incident 37136 failed: name 'find_candidates_for_incident' is not defined
Incident 37717 failed: name 'find_candidates_for_incident' is not defined
Incident 37879 failed: name 'find_candidates_for_incident' is not defined
Incident 37903 failed: name 'find_candidates_for_incident' is not defined
Incident 38008 failed: name 'find_candidates_for_incident' is not defined
Incident 38043 failed: name 'find_candidates_for_incident' is not defined
Incident 38124 failed: name 'find_candidates_for_incident' is not defined
Incident 38145 failed: name 'find_candidates_for_incident' is not defined
Incident 38148 failed: name 'find_candidates_for_incident' is not defined
Incident 38149 failed: name 'find_candidates_for_incident' is not defined
Incident 38172 failed: name 'find_candidates_for_incident